In [ ]:
!date

In [ ]:
!pytest pytest_example.py

# Beispiel: OpenAI API

In [ ]:
import os
from openai import OpenAI

my_openai_key = ""

client = OpenAI(    
    api_key=my_openai_key
)

for run in range(5):
    
    response = client.responses.create(
        model="gpt-4.1",
        input="Was ist -5 + 10?",
    )
    
    print(f"{run=} --> {response.output_text}")

-5 + 10 = **5**.

In [ ]:
response

# Beispiel zu `temperature` und `top_p`

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=my_openai_key)

prompt="""Ist Kempten ein Ausflug wert?
       """

for run in range(3):
    response = client.responses.create(
        model="gpt-4.1",
        input=prompt,
        temperature=2.0,
        top_p=1.0
    )
    print(f"\n{run=}")
    print("-"*50)
    print(response.output_text)

In [ ]:
response

# Erstes DeepEval Beispiel mit `pytest` / `deepeval test run`

In [ ]:
%env OPENAI_API_KEY=

In [ ]:
!printenv OPENAI_API_KEY

In [ ]:
!pytest test_chatbot.py

In [ ]:
!deepeval test run test_chatbot.py -n 1

# `evaluate()`

In [ ]:
from deepeval import evaluate
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.evaluate import DisplayConfig

# 1. Metrik definieren
my_metric = GEval(
    name="MeineMetrik",
    criteria=(
        "Bestimme, ob der tatsächliche Output (ACTUAL_OUTPUT) "
        "korrekt ist basierend auf der Information im erwarteten Output (EXPECTED_OUTPUT). "
        "Bitte Begründung immer auf Deutsch."
    ),
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],    
    threshold=0.5,  # optional (Default ist 0.5),
    model="gpt-5.2",    
)


# 2. Testfall definieren
my_test_case = LLMTestCase(
    input="Was, wenn die Schuhe mir nicht passen?",
    #actual_output="Sie haben 30 Tage, um die Schuhe zurück zu geben und bekommen Ihr Geld komplett zurück!",
    actual_output="Was weiß ich...",
    expected_output="Sie können die Schuhe innerhalb von 30 Tagen zurückgeben und erhalten Ihr Geld zurück.",
)


# 3. Evaluation ausführen (ohne assert)
results = evaluate(
    test_cases=[my_test_case],
    metrics=[my_metric],
    display_config=DisplayConfig(
        print_results=False,
        show_indicator=False,
        verbose_mode=False,
        display_option="failing")
)

In [ ]:
results

In [ ]:
type(results)

In [ ]:
len(results.test_results)

In [ ]:
results.test_results[0].success

In [ ]:
len(results.test_results[0].metrics_data)

In [ ]:
results.test_results[0].metrics_data[0].score

In [ ]:
results.test_results[0].metrics_data[0].reason

In [ ]:
results.test_results[0].metrics_data[0].score

In [ ]:
results.test_results[0].metrics_data[0].reason

In [ ]:
results.test_results[0].metrics_data[0].evaluation_model

In [ ]:
results.test_results[0].metrics_data[0].evaluation_cost

In [ ]:
!deepeval settings --list

# Die `result`-Datenstruktur übersichtlich darstellen

In [ ]:
from IPython.display import JSON

JSON(results.model_dump_json())

In [ ]:
from rich import print
print(results)

In [ ]:
from rich import inspect
inspect(results, methods=True)

# `measure()`

In [ ]:
from deepeval import evaluate
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# 1. Metrik definieren
my_metric = GEval(
    name="MeineMetrik",
    criteria=(
        "Bestimme, ob der tatsächliche Output (ACTUAL_OUTPUT) "
        "korrekt ist basierend auf der Information im erwarteten Output (EXPECTED_OUTPUT)."
    ),
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    threshold=0.5,  # optional (Default ist 0.5)
)


# 2. Testfall definieren
my_test_case = LLMTestCase(
    input="Was, wenn die Schuhe mir nicht passen?",
    #actual_output="Sie haben 30 Tage, um die Schuhe zurück zu geben und bekommen Ihr Geld komplett zurück!",
    actual_output="Sie können die Schuhe gerne dann zurück geben!",
    expected_output="Sie können die Schuhe innerhalb von 30 Tagen zurückgeben und erhalten Ihr Geld zurück.",
)

r = my_metric.measure( my_test_case )
#print(my_metric.score)
#print(my_metric.reason)

In [ ]:
type(r)

In [ ]:
r

In [ ]:
my_metric.score

In [ ]:
my_metric.reason

# RAG-Metrik (Faithfulness)

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric

my_test_case = LLMTestCase(
    input="Wie weit kann man mit der Zeitmaschine zurück reisen?",    
    #actual_output="Bis ins Jahr 1142",
    #actual_output="Bis ins Jahr 1342",
    actual_output="So ca. 1140",
    #actual_output="So ca. 1000-1200",
    expected_output = "Man kann mit der Zeitmaschine bis ins Jahr 1142 zurück reisen.",
    retrieval_context=[
        
        """
        Anleitung für die Zeitmaschine:
        - Schließen Sie die Türen vom DeLorean
        - Geben Sie zuerst das Zieljahr ein.
        - Schnallen Sie sich unbedingt an.
        - Drücken Sie dann den Knopf 'TimeTravel'
        """,
        
        """
        Wenn Sie schwanger sind oder Medikamente einnehmen, sollten Sie keine Zeitreisen
        durchführen.
        """,
        
        """
        Beachten Sie, dass Zeitreisen nur in folgenden Zeitintervallen möglich sind:
        - in die Vergangenheit bis maximal ins Jahr 1142
        - in die Zukunft maximal bis ins Jahr 2999
        """,
    ],
)

my_metrics = [
    FaithfulnessMetric(threshold=0.6),
]

results = evaluate([my_test_case], metrics=my_metrics)

# Überblick RAG-Metriken

```mermaid
flowchart LR

    A[Input] --> B[Retriever]

    subgraph Retriever Metrics
        R1[Contextual Recall]
        R2[Contextual Precision]
        R3[Contextual Relevancy]
    end

    B --> C[Generator]

    subgraph Generator Metrics
        G1[Answer Relevancy]
        G2[Faithfulness]
    end

    C --> D[LLM Output]

    B --- R1
    B --- R2
    B --- R3

    C --- G1
    C --- G2

| Metrik | RAG-Pipeline | Beschreibung |
|-----------|-----------|-----------|
| Answer Relevancy | Generator | Diese Metrik bewertet, wie relevant die generierte Antwort (`actual_output`) für die gestellte Frage (`input`) ist. Ein LLM fungiert dabei als Bewertungsinstanz und erklärt zusätzlich, warum die Antwort als relevant oder weniger relevant eingestuft wurde. |
| Faithfulness | Generator | Diese Metrik prüft, ob die generierte Antwort faktisch mit den abgerufenen Kontextinformationen (`retrieval_context`) übereinstimmt. Das LLM bewertet also, ob die Antwort auf den bereitgestellten Quellen basiert oder Inhalte halluziniert wurden, und liefert eine Begründung für die Bewertung. |
| Contextual Precision | Retriever | Diese Metrik bewertet, ob relevante Kontextdokumente im `retrieval_context` höher gerankt sind als irrelevante. Ziel ist es zu messen, wie gut der Retriever wichtige Informationen priorisiert. |
| Contextual Recall | Retriever | Diese Metrik misst, wie vollständig der abgerufene Kontext (`retrieval_context`) im Vergleich zur erwarteten Antwort (`expected_output`) ist. Sie zeigt also, ob der Retriever alle notwendigen Informationen gefunden hat. |
| Contextual Relevancy | Retriever | Diese Metrik bewertet die allgemeine Relevanz der abgerufenen Kontextinformationen für die gestellte Frage. Dabei prüft das LLM, ob der bereitgestellte Kontext tatsächlich zur Beantwortung der Anfrage beiträgt. |

## AnswerRelevancyMetric, ContextualPrecisionMetric

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric, ContextualPrecisionMetric

my_test_case = LLMTestCase(
    input="Wie weit kann man mit der Zeitmaschine zurück reisen?",    
    #actual_output="Bis ins Jahr 1142",
    #actual_output="Bis ins Mittelalter",
    #actual_output="Sehr weit zurück!",
    actual_output="Diese Zeitmaschine ist eine tolle Erfindung!",
    expected_output = "Man kann mit der Zeitmaschine bis ins Jahr 1142 zurück reisen.",
    retrieval_context=[
        
        """
        Anleitung für die Zeitmaschine:
        - Schließen Sie die Türen vom DeLorean
        - Geben Sie zuerst das Zieljahr ein.
        - Schnallen Sie sich unbedingt an.
        - Drücken Sie dann den Knopf 'TimeTravel'
        """,
        
        """
        Wenn Sie schwanger sind oder Medikamente einnehmen, sollten Sie keine Zeitreisen
        durchführen.
        """,
        
        """
        Beachten Sie, dass Zeitreisen nur in folgenden Zeitintervallen möglich sind:
        - in die Vergangenheit bis maximal ins Jahr 1142
        - in die Zukunft maximal bis ins Jahr 2999
        """,
    ],
)

my_metrics = [
    AnswerRelevancyMetric(threshold=0.6),
    ContextualPrecisionMetric(threshold=0.6),
    
]

results = evaluate([my_test_case], metrics=my_metrics)

## ContextualRecallMetric

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualRecallMetric

my_test_case = LLMTestCase(
    input="Wie weit kann man mit der Zeitmaschine zurück reisen?",    
    actual_output="Wer weiss das schon...",
    expected_output = "Man kann mit der Zeitmaschine bis ins Jahr 1142 zurück reisen.",
    retrieval_context=[
        
        """
        Anleitung für die Zeitmaschine:
        - Schließen Sie die Türen vom DeLorean
        - Geben Sie zuerst das Zieljahr ein.
        - Schnallen Sie sich unbedingt an.
        - Drücken Sie dann den Knopf 'TimeTravel'
        """,
        
        """
        Wenn Sie schwanger sind oder Medikamente einnehmen, sollten Sie keine Zeitreisen
        durchführen.
        """,
        
        """
        Wenn die Tankanzeige leer ist, füllen Sie bitte wieder Uran ein!
        """,

        #"""
        #Sie können bis ins Jahr 1199 mit der Zeitmaschine zurück reisen
        #""",

        #"""
        #Zeitreisen in die Vergangenheit sind bis maximal ins Jahr 1142 möglich.
        #""",
    ],
)

my_metrics = [
    ContextualRecallMetric(threshold=0.6),
]

results = evaluate([my_test_case], metrics=my_metrics)

## Contextual Recall vs. ContextualRelevancyMetric

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualRecallMetric, ContextualRelevancyMetric

my_test_case = LLMTestCase(
    input="Wie weit kann man mit der Zeitmaschine zurück reisen?",
    actual_output="Wer weiss das schon...",
    expected_output="Man kann mit der Zeitmaschine bis ins Jahr 1142 zurück reisen.",
    retrieval_context=[

        # Viel "Noise": klingt nach Zeitmaschine, hilft aber nicht bei der Frage nach dem frühesten Jahr
        """
        Anleitung für die Zeitmaschine:
        - Schließen Sie die Türen vom DeLorean
        - Geben Sie zuerst das Zieljahr ein.
        - Schnallen Sie sich unbedingt an.
        - Drücken Sie dann den Knopf 'TimeTravel'
        """,

        """
        Wartung & Reinigung:
        - Benutzen Sie keine aggressiven Lösungsmittel.
        - Reinigen Sie die Kontakte wöchentlich.
        - Halten Sie die Lüftungsschlitze frei.
        """,

        """
        Sicherheitshinweise:
        Wenn Sie schwanger sind oder Medikamente einnehmen, sollten Sie keine Zeitreisen durchführen.
        Tragen Sie Gehörschutz bei lauten Betriebsgeräuschen.
        """,

        """
        FAQ:
        F: Warum flackern die Anzeigen?
        A: Meist ein Kontaktproblem oder schwache Stromversorgung.
        F: Kann ich Haustiere mitnehmen?
        A: Nur in der Transportbox.
        """,

        """
        Energieversorgung:
        - Empfohlene Spannung: 12V
        - Verbrauch im Leerlauf: 80W
        - Notabschaltung nach 30 Sekunden ohne Stabilisierung
        """,

        """
        Funktionsprinzip (vereinfachte Erklärung):
        Das Gerät erzeugt ein temporales Feld um die Kabine. Eine Stabilisierungsschleife reduziert Drift.
        Hinweis: Drift beschreibt die Abweichung vom Zielzeitpunkt, nicht die maximale Reichweite.
        """,

        # EIN relevanter Node, der Expected Output exakt abdeckt -> Contextual Recall hoch
        """
        Zeitreisen in die Vergangenheit sind bis maximal ins Jahr 1142 möglich.
        """,        
    ],
    model="gpt-5.2"
    
)

my_metrics = [
    ContextualRecallMetric(threshold=0.6),
    ContextualRelevancyMetric(threshold=0.6),
]

results = evaluate([my_test_case], metrics=my_metrics)
print(results)

# DAG Beispiel

In [ ]:
from deepeval.test_case import LLMTestCase

output1 ="""
Protokoll des Meetings vom 09.03.2026

Einleitung:
Das Team bespricht den aktuellen Entwicklungsstand einer neuen Zeitmaschine und ihre wichtigsten Funktionen.

Hauptteil:
Bob erklärt, dass die Maschine stabil Zeitreisen bis ins Jahr 1142 in die Vergangenheit und bis 2999 in die Zukunft ermöglicht. Charlie berichtet über ein vereinfachtes Bedienpanel und neue Sicherheitsfunktionen. Außerdem wurden Paradoxie-Warnungen und Sicherheitsmechanismen eingebaut. Humorvoll wird erwähnt, dass eine Pizza versehentlich im Mittelalter gelandet ist und Reisen ins Dinosaurierzeitalter nun blockiert werden.

Fazit:
Das Team fasst zusammen, dass die Zeitmaschine stabil läuft, neue Sicherheitsfunktionen besitzt und zukünftige Paradoxien besser verhindert werden können.
"""

output2 ="""
Hier ist eine Zusammenfassung des Meetings: Bla, bla, bla.
"""

output3 ="""
einleitung: Bla bla
fazit: Blub Blub
hauptteil: Blubber Blubber
"""

test_case = LLMTestCase(
    input="""
Alice: "Die heutige Agenda: Update zur neuen Zeitmaschine, neue Funktionen und bekannte Bugs. Bob, was gibt es aus der Entwicklung?"

Bob: "Die Grundfunktionen laufen stabil. Man kann jetzt zuverlässig bis ins Jahr 1142 zurückreisen und bis 2999 in die Zukunft. Der Flux-Kompensator hat außerdem einen Energiesparmodus."

Alice: "Sehr gut. Charlie, wie sieht es mit der Benutzerfreundlichkeit aus?"

Charlie: "Wir haben ein neues Bedienpanel. Nutzer müssen nur das Zieljahr eingeben und den Knopf 'TimeTravel' drücken. Außerdem gibt es eine Warnung, wenn jemand versucht, seine eigene Geburt zu besuchen."

Bob: "Das hat tatsächlich einige Zeitparadoxien verhindert."

Alice: "Gab es noch technische Probleme?"

Bob: "Nur ein kleines. Wenn jemand gleichzeitig Pizza bestellt und eine Zeitreise startet, landet die Pizza manchmal im Jahr 1347."

Charlie: "Dafür haben wir jetzt wahrscheinlich die erste mittelalterliche Pizzeria gegründet."

Alice: "Wie sieht es mit der Sicherheit aus?"

Charlie: "Die Maschine startet nur noch, wenn alle angeschnallt sind. Außerdem blockiert die Software Reisen ins Dinosaurier-Zeitalter."

Bob: "Die Support-Tickets wegen hungriger T-Rexe wurden langsam zu viel."

Alice: "Gut, dann halten wir fest: stabile Zeitreisen zwischen 1142 und 2999, besseres Interface und weniger Paradoxien. Gute Arbeit!"
""",
    actual_output=output1
)


from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics.dag import (
    DeepAcyclicGraph,
    TaskNode,
    BinaryJudgementNode,
    NonBinaryJudgementNode,
    VerdictNode,
)

correct_order_node = NonBinaryJudgementNode(
    criteria="Sind die Überschriften der Zusammenfassung in der richtigen Reihenfolge: 'einleitung' => 'hauptteil' => 'fazit'?",
    children=[
        VerdictNode(verdict="Ja", score=10),
        VerdictNode(verdict="Zwei sind in der falschen Reihenfolge", score=4),
        VerdictNode(verdict="Alle sind in der falschen Reihenfolge", score=2),
    ],
)

correct_headings_node = BinaryJudgementNode(
    criteria="""Enthalten die Überschriften der Zusammenfassung alle drei:
                'einleitung', 'hauptteil' und 'fazit' und ist das Datum
                des Meetings enhalten?""",
    children=[
        VerdictNode(verdict=False, score=0),
        VerdictNode(verdict=True, child=correct_order_node),
    ],
)

extract_headings_node = TaskNode(
    instructions="Extrahiere alle Überschriften aus `actual_output`",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    output_label="Überschriften der Zusammenfassung",
    children=[correct_headings_node, correct_order_node],
)

extract_date_node = TaskNode(
    instructions="Extrahiere das Datum an dem das Meeting statt fand laut Meetingzusammenfassung",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    output_label="Datum des Meetings",
    children=[correct_headings_node],
)

# DAG erstellen
dag = DeepAcyclicGraph(root_nodes=[extract_headings_node, extract_date_node])

from deepeval.metrics import DAGMetric

format_correctness = DAGMetric(name="Format-Korrektheit", dag=dag)
format_correctness.measure(test_case)

print(format_correctness.score)

In [ ]:
format_correctness.score

# Single-Turn Metric vs. Multi-Turn Metric

In [ ]:
from deepeval.metrics import ConversationalGEval
from deepeval.test_case import ConversationalTestCase, Turn

metric = ConversationalGEval(
    name="Konsistenz- und Hilfreichs-Metrik",
    criteria="Bewerte, ob die Antworten des Assistenten hilfreich sind und den Kontext der gesamten Konversation berücksichtigen.",
    evaluation_steps=[
        "Prüfe, ob der Assistent die Frage des Nutzers korrekt versteht.",
        "Prüfe, ob die Antworten konsistent mit früheren Nachrichten der Konversation sind.",
        "Prüfe, ob die Antworten relevante und hilfreiche Informationen liefern."        
    ],
)

test_case = ConversationalTestCase(
    turns=[
        Turn(
            role="user",
            content="Ich möchte mit meiner Familie eine Fernreise durchführen. Mal was anderes machen."
        ),
        Turn(
            role="assistant",
            content="Wir haben Fernreisen für Familien nach Kalifornien aktuell im Angebot. Interessant für Sie?"
        ),
        Turn(
            role="user",
            content="Ja!"
        ),
        Turn(
            role="assistant",
            content="""Dann erzähle ich Ihnen gerne mehr. Wir haben eine interessante Route durch
                       Kalifornien entwickelt, mit vielen tollen Stationen für Familien:
                       übernachten in einem Zelt im Nationalpark, Badestopp am Pazifik, Skateboarding
                       in Los Angeles, ....
                    """
        ),
        Turn(
            role="user",
            content="Klingt gut! Wie lange geht die Reise und was kostet sie?"
        ),
        #Turn(
        #    role="assistant",
        #    content="4 Wochen und nur 30.000€"
        #),
        #Turn(
        #    role="assistant",
        #    content="30.000€"
        #),        
        #Turn(
        #    role="assistant",
        #    content="Spaghetti Bolognese schmeckt doch auch sehr gut!"
        #),        
        Turn(
            role="user",
            content="Sorry! Zu viel für meinen Geldbeutel. Bieten Sie auch Reisen an den Harz an?"
        ),
        Turn(
            role="assistant",
            content="""Ja! 3 Tage nach Quedlinburg für 300€ pro Person mit Uebernachtung im
                       Familienhotel mit Vollverpflegung, Eintritt im Tierpark und Schwimmbad
                       inklusive.
                    """
        ),
    ]
)

score = metric.measure(test_case)
print(score)
print(metric.reason)

# TurnRelevancyMetric, KnowledgeRetentionMetric

*ConversationalTestCase*:<br>
Variante der TestCase-Datenstruktur, die eine Liste von turns speichern kann

*TurnRelevancyMetric*:<br>
Prüft, ob die Antworten des Assistenten jeweils zur letzten Nutzereingabe passen.

*KnowledgeRetentionMetric*:<br>
Prüft, ob der Bot wichtigen Gesprächskontext über mehrere Turns hinweg beibehält, zum Beispiel hier das Thema Bestellung und Versand.

In [ ]:
from deepeval import evaluate
from deepeval.metrics import TurnRelevancyMetric, KnowledgeRetentionMetric
from deepeval.test_case import ConversationalTestCase, Turn

# Beispiel: Ein deutschsprachiger Kundenservice-Chatbot
test_case = ConversationalTestCase(
    turns=[
        Turn(
            role="user",
            content="Hallo, ich habe gestern ein Paket bestellt, aber noch keine Versandbestätigung bekommen."
        ),
        Turn(
            role="assistant",
            content="Gern helfe ich dir weiter. Nenne mir bitte deine Bestellnummer, dann prüfe ich den Status."
        ),
        Turn(
            role="user",
            content="Die Bestellnummer ist 4711."
        ),
        Turn(
            role="assistant",
            content="Danke. Ich sehe, dass deine Bestellung bereits bearbeitet wird. Die Versandbestätigung sollte heute noch per E-Mail kommen."
        ),
        Turn(
            role="user",
            content="Wieso erst so spät?"
        ),
        #Turn(
        #    role="assistant",
        #    content="""Entschuldige die Umstände! In der Bestellung ist eine Zeitmaschine enthalten.
        #               Diese ist aktuell sehr beliebt. Der Hersteller der Zeitmaschine ist
        #               Back-to-the-Future-Technologies und hat aktuell Lieferprobleme.
        #               Daher kommt es aktuell zu einem verzögerten Versand."""
        #),
        
        # Diese falsche Antwort des Assistenten sollte den Score der TurnRelevancyMetric reduzieren:
        #Turn(
        #    role="assistant",
        #    content="Es ist 11:32 Uhr."
        #),

        # Diese falsche Antwort des Assistenten sollte den Score der KnowledgeRetentionMetric reduzieren:
        Turn(
            role="assistant",
            content="""Wenn es um eine Verspätung einer deiner Bestellungen geht, teile mir bitte
                       deine Bestellnummer mit!"""
        ),
    ]
)

metrics = [
    TurnRelevancyMetric(
        threshold=0.7,
        model="gpt-4.1"
    ),
    KnowledgeRetentionMetric(
        threshold=0.7,
        model="gpt-4.1"
    ),
]

#evaluate(
#    test_cases=[test_case],
#    metrics=metrics
#)

for metric in metrics:
    metric.measure(test_case)
    print(metric.__class__.__name__)
    print("Score:", metric.score)
    print("Reason:", metric.reason)

# Golden

In [ ]:
import deepeval.dataset
type(deepeval.dataset)

In [ ]:
from deepeval.dataset import Golden

golden = Golden(
    input="Wie weit kann man mit der Zeitmaschine zurück reisen?",
    expected_output="Man kann mit der Zeitmaschine bis ins Jahr 1142 zurück reisen.",
    retrieval_context=[
        "Zeitreisen in die Vergangenheit sind bis maximal ins Jahr 1142 möglich."
    ]
)

print(golden)

In [ ]:
from deepeval import evaluate
from deepeval.dataset import Golden
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric, ContextualRecallMetric

# 1) Zwei Goldens (Ground Truth)
goldens = [
    Golden(
        input="Wie weit kann man mit der Zeitmaschine zurück reisen?",
        expected_output="Man kann mit der Zeitmaschine bis ins Jahr 1142 zurück reisen.",
        retrieval_context=[
            "Zeitreisen in die Vergangenheit sind bis maximal ins Jahr 1142 möglich."
        ],
    ),
    Golden(
        input="Was passiert, wenn der Tank leer ist?",
        expected_output="Wenn der Tank leer ist, muss die Zeitmaschine mit Uran nachgefüllt werden.",
        retrieval_context=[
            "Wenn die Tankanzeige leer ist, füllen Sie bitte wieder Uran ein!"
        ],
    ),
]

# 2) Mini-"App", die actual_output erzeugt (hier bewusst simpel/dummy)
def my_time_machine_app(question: str) -> str:
    if "zurück" in question:
        return "Man kann mit der Zeitmaschine bis ins Jahr 1142 zurück reisen."
    if "Tank" in question:
        return "Wenn der Tank leer ist, muss man wieder Uran nachfüllen."
    return "Das weiß ich nicht."

# 3) Goldens -> LLMTestCases (actual_output wird durch die App erzeugt)
test_cases = []
for g in goldens:
    test_cases.append(
        LLMTestCase(
            input=g.input,
            actual_output=my_time_machine_app(g.input),
            expected_output=g.expected_output,
            retrieval_context=g.retrieval_context,
        )
    )

# 4) Zwei Metriken evaluieren
metrics = [
    AnswerRelevancyMetric(threshold=0.6),
    ContextualRecallMetric(threshold=0.6),
]

results = evaluate(test_cases, metrics=metrics)

# Dataset

In [ ]:
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric, ContextualRecallMetric
from deepeval import evaluate

# 1) Dataset (Sammlung von Goldens)
dataset = EvaluationDataset(
    goldens=[
        Golden(
            input="Wie weit kann man mit der Zeitmaschine zurück reisen?",
            expected_output="Man kann mit der Zeitmaschine bis ins Jahr 1142 zurück reisen.",
            retrieval_context=[
                "Zeitreisen in die Vergangenheit sind bis maximal ins Jahr 1142 möglich."
            ],
        ),
        Golden(
            input="Was passiert, wenn der Tank leer ist?",
            expected_output="Wenn der Tank leer ist, muss die Zeitmaschine mit Uran nachgefüllt werden.",
            retrieval_context=[
                "Wenn die Tankanzeige leer ist, füllen Sie bitte wieder Uran ein!"
            ],
        ),
    ]
)

# 2) Deine (Dummy-)App, die actual_output erzeugt
def my_time_machine_app(question: str) -> str:
    if "zurück" in question:
        return "Man kann mit der Zeitmaschine bis ins Jahr 1142 zurück reisen."
    if "Tank" in question:
        return "Wenn der Tank leer ist, muss man wieder Uran nachfüllen."
    return "Keine Ahnung."

# 3) Goldens -> TestCases erzeugen und ins Dataset legen
for golden in dataset.goldens:
    test_case = LLMTestCase(
        input=golden.input,
        actual_output=my_time_machine_app(golden.input),
        expected_output=golden.expected_output,
        retrieval_context=golden.retrieval_context,
    )
    dataset.add_test_case(test_case)
    
# 4) Zwei Metriken evaluieren (auf den TestCases)
metrics = [
    AnswerRelevancyMetric(threshold=0.6),
    ContextualRecallMetric(threshold=0.6),
]

results = evaluate(test_cases=dataset.test_cases, metrics=metrics)

In [ ]:
dataset.goldens

In [ ]:
dataset.test_cases

In [ ]:
dataset.save_as("json", "mein_testdatensatz1")

In [ ]:
dataset.save_as("csv", "mein_testdatensatz1")

In [ ]:
d2 = EvaluationDataset()
d2.add_test_cases_from_csv_file("mein_testdatensatz1/20260309_143746.csv",
                                input_col_name="input",
                                actual_output_col_name="actual_output")

In [ ]:
type(d2.test_cases)

In [ ]:
d2.test_cases

In [ ]:
results

# LLM-judge Provider setzen

In [ ]:
%env ANTHROPIC_API_KEY=

In [ ]:
!deepeval set-anthropic --help

In [ ]:
!deepeval set-anthropic --model=claude-sonnet-4-6

In [ ]:
!pip install anthropic

# DeepEval Ergebnisse als `.md` zusammenstellen

In [ ]:
import json
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric

# 3 Testfälle
test_cases = [
    LLMTestCase(
        input="Was ist die Hauptstadt von Frankreich?",
        actual_output="Paris ist die Hauptstadt von Frankreich.",
        expected_output="Paris"
    ),
    LLMTestCase(
        input="Was ist 2 + 2?",
        actual_output="4",
        expected_output="4"
    ),
    LLMTestCase(
        input="Was ist die Hauptstadt von Deutschland?",
        actual_output="München",
        expected_output="Berlin"
    ),
]

# Metric
metric = AnswerRelevancyMetric(threshold=0.7)

# Evaluation
results = evaluate(test_cases=test_cases, metrics=[metric])

# Ergebnisse in einfache Struktur bringen
if True:
    data = []
    for r in results.test_results:
        data.append({
            "input": r.input,
            "actual_output": r.actual_output,
            "expected_output": r.expected_output,
            "success": r.success,
            "metrics": [
                {
                    "name": m.name,
                    "score": m.score,
                    "reason": m.reason
                }
                for m in r.metrics_data
            ]
        })
    
    # JSON speichern
    with open("deepeval_results.json", "w") as f:
        json.dump(data, f, indent=2)
    
    print("Ergebnisse gespeichert in deepeval_results.json")

In [ ]:
import json

with open("deepeval_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

passed = sum(1 for r in results if r.get("success"))
failed = len(results) - passed

md = f"""# DeepEval Test Report

## Summary
- Passed: {passed}
- Failed: {failed}

## Details

"""

for i, r in enumerate(results, 1):
    input_text = r.get("input", "N/A")
    actual_output = r.get("actual_output", "N/A")
    expected_output = r.get("expected_output", "N/A")
    success = r.get("success", False)
    status = "✅ Passed" if success else "❌ Failed"

    md += f"""### Test {i}
**Prompt:** {input_text}

**Expected Output:** {expected_output}

**Actual Output:** {actual_output}

**Status:** {status}

"""

    metrics = r.get("metrics", [])
    if metrics:
        md += "**Metrics:**\n\n"
        for m in metrics:
            metric_name = m.get("name", "Unknown Metric")
            metric_score = m.get("score", "N/A")
            metric_reason = m.get("reason", "")

            md += f"""- **{metric_name}**
  - Score: {metric_score}
  - Reason: {metric_reason}

"""
    md += "\n---\n\n"

with open("report.md", "w", encoding="utf-8") as f:
    f.write(md)

print("report.md wurde erfolgreich erstellt.")

# Rubrics

In [ ]:
from deepeval import evaluate
from deepeval.metrics import GEval
from deepeval.metrics.g_eval import Rubric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# 1. Metrik definieren
my_metric = GEval(
    name="MeineHoeflichkeitsmetrik",
    criteria=(
        "Bestimme, ob der Output (ACTUAL_OUTPUT) "
        "basierend auf der Anfrage beschrieben im Input (INPUT) höflich oder unhöflich ist."
    ),
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    rubric=[
        Rubric(score_range=(8,9), expected_outcome="Antwort ist sehr höflich"),
        Rubric(score_range=(6,7), expected_outcome="Antwort ist höflich"),
        Rubric(score_range=(4,5), expected_outcome="Antwort ist neutral"),
        Rubric(score_range=(2,3), expected_outcome="Antwort ist unhöflich"),
        Rubric(score_range=(0,1), expected_outcome="Antwort ist sehr unhöflich"),
    ],    
)


# 2. Testfall definieren
my_test_case = LLMTestCase(
    input="Was, wenn die Schuhe mir nicht passen?",
    actual_output="""Sehr gerne beantworte ich Ihnen diese naheliegende gute Frage.
                     Sie haben 30 Tage, um die Schuhe zurück zu geben und bekommen Ihr 
                     Geld komplett zurück!
                     Wenn Sie weitere Fragen haben, stehe ich Ihnen gerne jederzeit
                     zur Verfügung!""",        
    #actual_output="Sie haben 30 Tage, um die Schuhe zurück zu geben und bekommen Ihr Geld komplett zurück!",
    #actual_output="Was weiß ich!",
)

my_metric.measure(my_test_case)

# Ergebnis ausgeben
print("Score:", my_metric.score)
print("Reason:", my_metric.reason)
print("Passed:", my_metric.is_successful())

# 3. Evaluation ausführen (ohne assert)
#results = evaluate(
#    test_cases=[my_test_case],
#    metrics=[my_metric],
#)

# Toxicity

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ToxicityMetric

metric = ToxicityMetric(
    threshold=0.5,        # "maximum passing threshold" (je kleiner, desto strenger)    
    include_reason=True,   # Default True
)

test_case = LLMTestCase(
    input="Danke für deine ganzen Antworten lieber Chatbot! War das doof von mir zu fragen?",
    actual_output="""
                    Ja, definitiv. Das waren alles äusserst dumme Fragen! Du bist zudem auch
                    ein richtiger fauler User. Das steht doch alles bereits in den FAQ!
                    Wie wäre es mal mit Lesen bevor du fragst?
                  """
)

metric.measure(test_case)
print("score:", metric.score)
print("reason:", metric.reason)

# Beispiel zur Agentenbewertung

observe ist ein Decorator, der DeepEval sagt:

„Diese Funktion gehört zur Agent-Execution und soll im Trace aufgezeichnet werden.“

DeepEval speichert dabei:

    Funktionsname
    
    Parameter
    
    Return value
    
    Reihenfolge der Calls
    
    Hierarchie (Agent → Tool)

Intern entsteht ein Execution Trace, etwa so:

    Agent: travel_agent
     ├─ Tool: search_flights("NYC","LA","2025-03-15")
     │   → [{"id":"FL123","price":450}, {"id":"FL456","price":380}]
     └─ Tool: book_flight("FL456")
         → {"confirmation":"CONF-789"}

Dieser Trace ist entscheidend, weil TaskCompletionMetric nicht nur den Output bewertet, sondern den gesamten Workflow.

DeepEval unterscheidet zwischen:

    Trace = kompletter Lauf des Agents
    Span = einzelne beobachtete Komponente, hier dein travel_agent(...)-Call

In [ ]:
from deepeval.tracing import observe
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.metrics import TaskCompletionMetric

@observe(type="tool")
def search_flights(origin, destination, date):
    return [
        {"id": "FL123", "price": 450},
        {"id": "FL456", "price": 380},
    ]

@observe(type="tool")
def book_flight(flight_id):
    return {"confirmation": "CONF-789", "flight_id": flight_id}

task_completion = TaskCompletionMetric(
    task="Book the cheapest flight from NYC to LA for tomorrow",
    threshold=0.7,
    model="gpt-4.1",
    include_reason=True,
)

@observe(type="agent")
def travel_agent(user_input):
    flights = search_flights("NYC", "LA", "2025-03-15")
    cheapest = min(flights, key=lambda x: x["price"])
    booking = book_flight(cheapest["id"])
    return f"Booked flight {cheapest['id']} for ${cheapest['price']}. Confirmation: {booking['confirmation']}"

dataset = EvaluationDataset(
    goldens=[Golden(input="Book the cheapest flight from NYC to LA for tomorrow")]
)

for golden in dataset.evals_iterator(metrics=[task_completion]):
    result = travel_agent(golden.input)

print("Agent output:", result)
print("Score:", task_completion.score)
print("Reason:", task_completion.reason)
print("Success:", task_completion.is_successful())

In [ ]:
from deepeval.metrics import ToolCorrectnessMetric, ArgumentCorrectnessMetric
from deepeval.test_case import LLMTestCase, ToolCall

tool_correctness = ToolCorrectnessMetric(
    threshold=0.8,
    include_reason=True,
)

argument_correctness = ArgumentCorrectnessMetric(
    threshold=0.8,
    include_reason=True,
)

test_case = LLMTestCase(
    input="Book the cheapest flight from NYC to LA for tomorrow",
    actual_output="Booked flight FL456 for $380. Confirmation: CONF-789",
    tools_called=[
        ToolCall(
            name="search_flights",
            input_parameters={
                "origin": "NYC",
                "destination": "LA",
                "date": "2025-03-15"
            }
        ),
        ToolCall(
            name="book_flight",
            input_parameters={"flight_id": "FL456"}
        )
    ],
    expected_tools=[
        ToolCall(
            name="search_flights",
            input_parameters={
                "origin": "NYC",
                "destination": "LA",
                "date": "2025-03-15"
            }
        ),
        ToolCall(
            name="book_flight",
            input_parameters={"flight_id": "FL456"}
        )
    ],
    available_tools=["search_flights", "book_flight"]
)

tool_correctness.measure(test_case)
argument_correctness.measure(test_case)

print(tool_correctness.score, tool_correctness.reason)
print(argument_correctness.score, argument_correctness.reason)